# Loading PEC CSV Data

This notebook shows a minimal PEC workflow with the built-in `pec_csv` loader. It uses the example export that ships with `cellpy`, inspects the raw file header, loads the cycling data, and generates step and summary tables.

In [ ]:
from pathlib import Path
import sys

# Changes to the cellpy repo can directly be used without installing the package. This is useful for development and testing.
repo_root = next(
    (path for path in [Path.cwd(), Path.cwd().parent] if (path / "cellpy" / "__init__.py").exists()),
    None,
)
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
# IF not using the cellpy repo, make sure to install cellpy in the current environment (e.g., pip install cellpy) and comment the above code.
import cellpy

In [ ]:
local_pec_file = Path("data/pec.csv").resolve()
# local_pec_file = Path("../local/raw_data/Test25196.csv").resolve()
if local_pec_file.is_file():
    pec_file = local_pec_file
    print(f"Using local PEC file: {pec_file}")
else:
    from cellpy.utils import example_data

    pec_file = example_data.pec_file_path()
    print(f"Using example PEC file: {pec_file}")

pec_file

In [ ]:
# Let's take a look at the first few lines of the PEC file to understand its structure.
with open(pec_file, encoding="utf-8-sig") as handle:
    for line_number, line in zip(range(1, 13), handle):
        print(f"{line_number:02d}: {line.rstrip()}")

In [29]:
# Now we can load the PEC data using cellpy. We specify the instrument type as "pec_csv" and set the mass to 1.0 (this is a parameter that can be adjusted based on the specific dataset).
cell = cellpy.get(
    filename=pec_file,
    instrument="pec_csv",
    mass=1.0,
    auto_summary=False,
)
cell

/home/meg/projects/cellpy/cellpy/readers/instruments/pec_csv.py:153: UserWarning: raw limits have not been subject for testing yet
  warnings.warn("raw limits have not been subject for testing yet")


,data_point,test_id,cell,shelf,position,step_index,cycle_index,test_time,step_time,date_time,...,temp_min_test_[var12]_ma,volt_limit_upper_[var14]_ma,volt_max_[var8]_ohm,volt_max_extra_[var11],volt_min_[var9]_mah,volt_pulse_max_[var16]_ah,volt_pulse_min_[var17]_a,station_temperature_degc,sub_step_index,sub_step_time
count,17839.000000,17839.0,17839.0,17839.0,17839.0,17839.000000,17839.000000,17839.000000,17839.000000,17839,...,2.00000,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,0.0,17839.0,17839.0
mean,8920.000000,25196.0,1.0,14.0,1.0,163.533999,1.989013,30883.618760,423.764175,2026-01-07 02:30:39.618308096,...,757.50000,2.100210e+07,2.100210e+07,2.150215e+07,1.500150e+07,2.075208e+07,1.600160e+07,NaN,0.0,0.0
min,1.000000,25196.0,1.0,14.0,1.0,0.000000,1.000000,0.005000,0.001000,2026-01-06 17:55:56,...,15.00000,4.200000e+03,4.200000e+03,4.300000e+03,3.000000e+03,4.150000e+03,3.200000e+03,NaN,0.0,0.0
25%,4460.500000,25196.0,1.0,14.0,1.0,144.000000,2.000000,22725.903500,0.021000,2026-01-07 00:14:42,...,386.25000,1.050315e+07,1.050315e+07,1.075322e+07,7.502250e+06,1.037811e+07,8.002400e+06,NaN,0.0,0.0
50%,8920.000000,25196.0,1.0,14.0,1.0,150.000000,2.000000,30666.619000,0.042000,2026-01-07 02:27:03,...,757.50000,2.100210e+07,2.100210e+07,2.150215e+07,1.500150e+07,2.075208e+07,1.600160e+07,NaN,0.0,0.0
75%,13379.500000,25196.0,1.0,14.0,1.0,174.000000,2.000000,39457.500000,2.500000,2026-01-07 04:53:33.500000,...,1128.75000,3.150105e+07,3.150105e+07,3.225108e+07,2.250075e+07,3.112604e+07,2.400080e+07,NaN,0.0,0.0
max,17839.000000,25196.0,1.0,14.0,1.0,360.000000,3.000000,58264.470000,11762.365000,2026-01-07 10:07:00,...,1500.00000,4.200000e+07,4.200000e+07,4.300000e+07,3.000000e+07,4.150000e+07,3.200000e+07,NaN,0.0,0.0
std,5149.820062,0.0,0.0,0.0,0.0,53.061432,0.341462,11233.829671,1557.545001,NaN,...,1050.05357,2.969551e+07,2.969551e+07,3.040255e+07,2.121108e+07,2.934200e+07,2.262515e+07,NaN,0.0,0.0
,data_point,test_id,cell,rack,shelf,position,cell_id,step_index,instruction_name,cycle_index,...,temp_min_test_[var12]_ma,volt_limit_upper_[var14]_ma,volt_max_[var8]_ohm,volt_max_extra_[var11],volt_min_[var9]_mah,volt_pulse_max_[var16]_ah,volt_pulse_min_[var17]_a,station_temperature_degc,sub_step_index,sub_step_time
0,1,25196,1,ACT4,14,1,345626_9356111_33_6,0,Idle,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0


In [30]:
# Now that we have loaded the data, we can take a look at the raw data to see what it looks like. The raw data is stored in a pandas DataFrame, and we can use the `head()` method to see the first few rows.
raw = cell.data.raw
raw[["data_point", "cycle_index", "step_index", "test_time", "current", "voltage"]].head()

,data_point,cycle_index,step_index,test_time,current,voltage
0,1,1,0,0.005,-0.000001,3.569225
1,2,1,8,0.010,0.000002,3.568968
2,3,1,12,0.015,-0.000001,3.569144
3,4,1,19,0.016,0.000004,3.569336
4,5,1,23,0.021,-0.000002,3.568866


In [ ]:
# After loading the data, we need to make the step table and summary to have access to the cycle and step information.
cell.make_step_table()
cell.make_summary()

print(f"Loaded rows: {len(cell.data.raw)}")
print(f"Cycles: {cell.get_cycle_numbers()[:10]}")
print(f"Start time: {cell.data.start_datetime}")

In [ ]:
cell.data.steps.head()

In [ ]:
cell.data.summary.head()

In [ ]:
# Save in h5 format
cell.save("pec_data.h5")